In [2]:
!pip install scikit-image

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   --------------------- ------------------ 6.6/12.0 MB 39.0 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 33.7 MB/s  0:00:00

   ---------------------------------------- 0/4 [tifffile]
   -------------------- ------------------- 2/4 [imageio]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ---------------------------------------- 4/4 [scikit-image]




[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# model_comparison.py
#
# Stage 8: compares classical models and a small CNN on the signature
# legibility labels, using ONLY the dev rows of splits.csv and the existing
# signer-grouped folds (cv_seed42 / cv_seed43 / cv_seed44).
# The test rows are never loaded into any model.
#
# Models
#   baseline : majority-class predictor
#   shortcut : logistic regression on contrast alone (the bar to beat)
#   classical: logistic regression, random forest, SVM (RBF), gradient boosting
#              on HOG + shape features from intensity-normalized images
#   neural   : small CNN, run on raw images and on intensity-normalized images
#
# Metrics: PR-AUC (primary), ROC-AUC, F1 at the default threshold, pooled and
# per source family.
#
# Memory: all 1,289 images are cached once as uint8 (about 85 MB at 256 px,
# about 21 MB at 128 px). The CNN is trained in mini-batches and each batch is
# moved to the GPU separately, so peak GPU use is a few hundred MB.
#
# Needs in the working folder: splits.csv, signatures_raw.xlsx
#   (the images come from the Hugging Face dataset the first time; after that
#    they are read from signature_images_256.npy)
# Outputs: model_comparison_summary.csv, model_comparison_by_family.csv,
#          model_comparison_folds.csv, model_comparison_oof.csv
# Requires: pandas numpy scikit-learn scikit-image scipy pillow datasets
#           openpyxl torch (torch only for the CNN)
#
# Usage:
#   python model_comparison.py --quick            # seed 42 only, fast check
#   python model_comparison.py                    # all 3 seeds x 5 folds
#   python model_comparison.py --skip-cnn         # classical models only
#   python model_comparison.py --skip-classical   # CNN only
#   python model_comparison.py --batch-size 16 --img-size 96   # if GPU memory is tight

import argparse
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from skimage.feature import hog
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=UserWarning)

In [14]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    # Small test on the GPU
    x = torch.rand(1000, 1000, device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    print("Matmul OK, result on:", y.device)
    print(f"Memory allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB")
else:
    print("No GPU detected. Running on CPU.")

PyTorch: 2.15.0+cu134
CUDA available: True
CUDA version: 13.4
GPU count: 1
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU
Matmul OK, result on: cuda:0
Memory allocated: 8.0 MB


In [18]:

DATASET = "sl4shuur/handwriting-signatures-dataset"
CV_SEEDS = (42, 43, 44)
BLANK_RANGE = 20          # images with a darkest-to-lightest range below this are blank

# ----------------------------------------------------------------------------
# Loading and caching images
# ----------------------------------------------------------------------------
def load_images(workdir, splits):
    cache = workdir / "signature_images_256.npy"
    if cache.exists():
        arr = np.load(cache)
        if len(arr) == len(splits):
            print(f"Using cached images from {cache.name} {arr.shape}")
            return arr
    from datasets import load_dataset, disable_progress_bar
    disable_progress_bar()
    print("Loading images from Hugging Face (first run only) ...")
    ds = load_dataset(DATASET, split="train")
    xlsx = workdir / "signatures_raw.xlsx"
    if xlsx.exists():
        meta = pd.read_excel(xlsx)
        if list(meta["file name"]) != list(splits["file name"]):
            raise RuntimeError("splits.csv rows are not in the spreadsheet order.")
        if list(meta["text"]) != list(ds["text"]):
            raise RuntimeError("Spreadsheet caption order differs from the dataset order.")
    if len(ds) != len(splits):
        raise RuntimeError(f"Dataset has {len(ds)} rows, splits.csv has {len(splits)}.")
    out = np.zeros((len(ds), 256, 256), dtype=np.uint8)
    for i in range(len(ds)):
        im = ds[i]["image"].convert("L")
        if im.size != (256, 256):
            im = im.resize((256, 256), Image.BOX)
        out[i] = np.asarray(im)
    np.save(cache, out)
    print(f"Saved {cache.name} {out.shape}")
    return out


# ----------------------------------------------------------------------------
# Preprocessing variants. Every variant returns uint8 with INK = HIGH, paper = 0.
#   raw  : only inverted, keeps the source ink darkness (the shortcut)
#   norm : each image stretched to its own min-max range
#   bin  : stretched, then binarized so every stroke is full black on white
# ----------------------------------------------------------------------------
def make_variant(imgs256, variant, size):
    out = np.zeros((len(imgs256), size, size), dtype=np.uint8)
    for i, g in enumerate(imgs256):
        g = g.astype(np.float32)
        lo, hi = g.min(), g.max()
        if variant == "raw":
            x = 1.0 - g / 255.0
        else:
            if hi - lo < BLANK_RANGE:           # blank image: do not amplify noise
                x = np.zeros_like(g)
            else:
                x = 1.0 - (g - lo) / (hi - lo)
            if variant == "bin":
                x = (x > 0.5).astype(np.float32)
        im = Image.fromarray((x * 255).astype(np.uint8))
        if size != 256:
            im = im.resize((size, size), Image.BOX)
        out[i] = np.asarray(im)
    return out


# ----------------------------------------------------------------------------
# Classical features: HOG + shape features from normalized images.
# Contrast is deliberately NOT included (it is the known shortcut).
# ----------------------------------------------------------------------------
def shape_features(u8):
    x = u8.astype(np.float32) / 255.0
    ink = x > 0.5
    h, w = ink.shape
    rows, cols = np.where(ink.any(axis=1))[0], np.where(ink.any(axis=0))[0]
    if len(rows) == 0:
        return [0.0] * 8
    bh = (rows[-1] - rows[0] + 1) / h
    bw = (cols[-1] - cols[0] + 1) / w
    ys, xs = np.nonzero(ink)
    ncomp = ndimage.label(ink)[1]
    return [ink.mean(), bw, bh, bw * bh, xs.mean() / w, ys.mean() / h,
            ink.sum() / max(1.0, bw * bh * h * w), float(ncomp)]


def classical_features(norm_u8):
    feats = []
    for u8 in norm_u8:
        x = u8.astype(np.float32) / 255.0
        hg = hog(x, orientations=9, pixels_per_cell=(16, 16),
                 cells_per_block=(2, 2), feature_vector=True)
        feats.append(np.concatenate([hg, shape_features(u8)]))
    return np.asarray(feats, dtype=np.float32)


def contrast_feature(imgs256):
    g = imgs256.reshape(len(imgs256), -1).astype(np.float32)
    bg = np.percentile(g, 95, axis=1)
    return (bg - g.min(axis=1)).reshape(-1, 1)


def classical_models():
    return {
        "Majority baseline": lambda: DummyClassifier(strategy="prior"),
        "Logistic (contrast only)": lambda: make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=1000)),
        "Logistic regression": lambda: make_pipeline(
            StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=3000)),
        "Random forest": lambda: RandomForestClassifier(
            n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample",
            n_jobs=-1, random_state=0),
        "SVM (RBF)": lambda: make_pipeline(
            StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
        "Gradient boosting": lambda: HistGradientBoostingClassifier(
            max_depth=3, max_iter=200, learning_rate=0.05, class_weight="balanced",
            random_state=0),
    }


def score_and_threshold(model, X):
    """Returns (continuous score, decision threshold for F1)."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1], 0.5
    return model.decision_function(X), 0.0


# ----------------------------------------------------------------------------
# CNN (PyTorch imported lazily so the classical part runs without it)
# ----------------------------------------------------------------------------
def run_cnn_fold(Xtr, ytr, Xva, args, device, seed):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    torch.manual_seed(seed)
    np.random.seed(seed)

    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                             nn.ReLU(inplace=True), nn.MaxPool2d(2))

    model = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                          nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3),
                          nn.Linear(128, 1)).to(device)
    pos = max(1, int(ytr.sum()))
    pos_weight = torch.tensor([(len(ytr) - pos) / pos], device=device, dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=3e-3, total_steps=args.epochs * int(np.ceil(len(ytr) / args.batch_size)))
    use_amp = device.type == "cuda" and not args.no_amp
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    Xtr_t = torch.from_numpy(Xtr)          # uint8 on CPU; moved to GPU one batch at a time
    ytr_t = torch.from_numpy(ytr.astype(np.float32))
    Xva_t = torch.from_numpy(Xva)

    def augment(xb):
        # small random shift / scale / rotation; empty space stays paper (0)
        n = xb.shape[0]
        ang = (torch.rand(n, device=xb.device) - 0.5) * 2 * np.deg2rad(3)
        sc = 1 + (torch.rand(n, device=xb.device) - 0.5) * 0.10
        tx = (torch.rand(n, device=xb.device) - 0.5) * 0.08
        ty = (torch.rand(n, device=xb.device) - 0.5) * 0.08
        theta = torch.zeros(n, 2, 3, device=xb.device)
        theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = sc * torch.cos(ang), -sc * torch.sin(ang), tx
        theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sc * torch.sin(ang), sc * torch.cos(ang), ty
        grid = F.affine_grid(theta, xb.shape, align_corners=False)
        return F.grid_sample(xb, grid, padding_mode="zeros", align_corners=False)

    for _ in range(args.epochs):            # fixed epoch count: no peeking at the validation fold
        model.train()
        perm = torch.randperm(len(ytr_t))
        for s in range(0, len(perm), args.batch_size):
            idx = perm[s:s + args.batch_size]
            if len(idx) < 2:
                continue
            xb = (Xtr_t[idx].to(device).float() / 255.0).unsqueeze(1)
            yb = ytr_t[idx].to(device)
            xb = augment(xb)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = loss_fn(model(xb).squeeze(1), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

    model.eval()
    out = []
    with torch.no_grad():
        for s in range(0, len(Xva_t), args.batch_size * 2):
            xb = (Xva_t[s:s + args.batch_size * 2].to(device).float() / 255.0).unsqueeze(1)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out.append(torch.sigmoid(model(xb).squeeze(1)).float().cpu().numpy())
    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(out)


# ----------------------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------------------
def fold_metrics(y, score, thr):
    return {"pr_auc": average_precision_score(y, score),
            "roc_auc": roc_auc_score(y, score),
            "f1": f1_score(y, (score > thr).astype(int), zero_division=0)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--workdir", default=".")
    ap.add_argument("--quick", action="store_true", help="seed 42 only")
    ap.add_argument("--skip-cnn", action="store_true")
    ap.add_argument("--skip-classical", action="store_true")
    ap.add_argument("--img-size", type=int, default=128)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--cnn-variants", default="raw,norm", help="comma list from raw,norm,bin")
    ap.add_argument("--no-amp", action="store_true")
    args, _ = ap.parse_known_args()

    wd = Path(args.workdir)
    splits = pd.read_csv(wd / "splits.csv")
    imgs = load_images(wd, splits)

    dev_mask = (splits["split"] == "dev").to_numpy()
    dev = splits[dev_mask].reset_index(drop=True)       # test rows are dropped here
    assert (dev["split"] == "dev").all()
    imgs_dev = imgs[dev_mask]
    y = dev["label"].to_numpy().astype(int)
    seeds = (42,) if args.quick else CV_SEEDS
    print(f"Dev rows: {len(dev)}, legible {y.mean():.3f}; seeds {seeds}")

    folds, oof = [], {}

    def record(name, seed, fold, yv, score, thr, fam_index):
        m = fold_metrics(yv, score, thr)
        folds.append({"model": name, "seed": seed, "fold": fold, **m})
        oof.setdefault((name, seed), np.full(len(dev), np.nan))[fam_index] = score

    # ---- classical models --------------------------------------------------
    if not args.skip_classical:
        t0 = time.time()
        print("\nExtracting HOG + shape features from normalized images ...")
        norm = make_variant(imgs_dev, "norm", args.img_size)
        X_feat = classical_features(norm)
        X_contrast = contrast_feature(imgs_dev)
        print(f"  feature matrix {X_feat.shape} in {time.time() - t0:.0f}s")
        models = classical_models()
        for seed in seeds:
            col = f"cv_seed{seed}"
            for k in sorted(dev[col].unique()):
                va = (dev[col] == k).to_numpy()
                tr = ~va
                for name, make in models.items():
                    X = X_contrast if name == "Logistic (contrast only)" else X_feat
                    m = make().fit(X[tr], y[tr])
                    score, thr = score_and_threshold(m, X[va])
                    record(name, seed, k, y[va], score, thr, np.where(va)[0])
            print(f"  classical models done for seed {seed} ({time.time() - t0:.0f}s)")

    # ---- CNN ----------------------------------------------------------------
    if not args.skip_cnn:
        try:
            import torch
        except ImportError:
            print("\nPyTorch is not installed; skipping the CNN.")
            torch = None
        if torch is not None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            print(f"\nCNN on {device}"
                  + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else
                     " (no GPU found: this will be slow)"))
            for variant in [v.strip() for v in args.cnn_variants.split(",")]:
                name = f"CNN ({variant})"
                Xv = make_variant(imgs_dev, variant, args.img_size)
                t0 = time.time()
                for seed in seeds:
                    col = f"cv_seed{seed}"
                    for k in sorted(dev[col].unique()):
                        va = (dev[col] == k).to_numpy()
                        tr = ~va
                        try:
                            score = run_cnn_fold(Xv[tr], y[tr], Xv[va], args, device, seed * 10 + int(k))
                        except torch.cuda.OutOfMemoryError:
                            raise SystemExit("GPU out of memory: rerun with --batch-size 16 "
                                             "and/or --img-size 96.")
                        record(name, seed, k, y[va], score, 0.5, np.where(va)[0])
                    print(f"  {name} seed {seed} done ({time.time() - t0:.0f}s)")

    # ---- summaries ----------------------------------------------------------
    fdf = pd.DataFrame(folds)
    fdf.to_csv(wd / "model_comparison_folds.csv", index=False)
    summ = fdf.groupby("model")[["pr_auc", "roc_auc", "f1"]].agg(["mean", "std"])
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    summ = summ.sort_values("pr_auc_mean", ascending=False)
    summ.round(4).to_csv(wd / "model_comparison_summary.csv")

    rows = []
    for (name, seed), sc in oof.items():
        for fam in sorted(dev["source_family"].unique()):
            msk = (dev["source_family"] == fam).to_numpy()
            if len(np.unique(y[msk])) == 2 and msk.sum() >= 30:
                rows.append({"model": name, "seed": seed, "family": fam, "n": int(msk.sum()),
                             "pr_auc": average_precision_score(y[msk], sc[msk]),
                             "roc_auc": roc_auc_score(y[msk], sc[msk])})
    fam_df = (pd.DataFrame(rows).groupby(["model", "family"])
              .agg(n=("n", "first"), pr_auc=("pr_auc", "mean"), roc_auc=("roc_auc", "mean"))
              .reset_index())
    fam_df.round(4).to_csv(wd / "model_comparison_by_family.csv", index=False)
    pd.DataFrame({f"{n}__s{s}": v for (n, s), v in oof.items()}).assign(
        file_name=dev["file name"], label=y, family=dev["source_family"]
    ).to_csv(wd / "model_comparison_oof.csv", index=False)

    chance = y.mean()
    print(f"\n{'=' * 78}\nPOOLED RESULTS: mean +/- sd over {len(seeds)} seed(s) x 5 grouped folds "
          f"(dev rows only)\nChance: PR-AUC {chance:.3f}, ROC-AUC 0.500\n{'=' * 78}")
    print(f"{'model':28s} {'PR-AUC':>15s} {'ROC-AUC':>15s} {'F1':>15s}")
    for name, r in summ.iterrows():
        print(f"{name:28s} {r.pr_auc_mean:.3f} +/- {r.pr_auc_std:.3f}  "
              f"{r.roc_auc_mean:.3f} +/- {r.roc_auc_std:.3f}  {r.f1_mean:.3f} +/- {r.f1_std:.3f}")
    if "Logistic (contrast only)" in summ.index:
        bar = summ.loc["Logistic (contrast only)", "pr_auc_mean"]
        print(f"\nShortcut bar (contrast only): PR-AUC {bar:.3f}. Models within about one fold "
              f"sd (~0.07) of this are not clearly learning legibility.")

    print(f"\n{'=' * 78}\nPER SOURCE FAMILY (out-of-fold, averaged over seeds)\n{'=' * 78}")
    for fam, sub in fam_df.groupby("family"):
        base = y[(dev['source_family'] == fam).to_numpy()].mean()
        print(f"\nFamily {fam}: n={int(sub['n'].iloc[0])}, legible rate {base:.3f}")
        print(sub.sort_values("pr_auc", ascending=False)[["model", "pr_auc", "roc_auc"]]
              .round(3).to_string(index=False))
    print("\nSaved model_comparison_summary.csv, _by_family.csv, _folds.csv, _oof.csv")


In [19]:
if __name__ == "__main__":
    main()

Loading images from Hugging Face (first run only) ...


Saved signature_images_256.npy (1289, 256, 256)
Dev rows: 1035, legible 0.290; seeds (42, 43, 44)

Extracting HOG + shape features from normalized images ...
  feature matrix (1035, 1772) in 1s
  classical models done for seed 42 (348s)
  classical models done for seed 43 (714s)
  classical models done for seed 44 (1073s)

CNN on cuda (NVIDIA GeForce RTX 5060 Laptop GPU)
  CNN (raw) seed 42 done (29s)
  CNN (raw) seed 43 done (53s)
  CNN (raw) seed 44 done (76s)
  CNN (norm) seed 42 done (24s)
  CNN (norm) seed 43 done (48s)
  CNN (norm) seed 44 done (72s)

POOLED RESULTS: mean +/- sd over 3 seed(s) x 5 grouped folds (dev rows only)
Chance: PR-AUC 0.290, ROC-AUC 0.500
model                                 PR-AUC         ROC-AUC              F1
Logistic (contrast only)     0.453 +/- 0.083  0.652 +/- 0.088  0.150 +/- 0.128
SVM (RBF)                    0.417 +/- 0.099  0.625 +/- 0.104  0.388 +/- 0.108
CNN (raw)                    0.411 +/- 0.084  0.615 +/- 0.057  0.332 +/- 0.135
CNN (norm